# LensWord — HuggingFace Model Comparison (Fixed)

This notebook compares our LSTM against three pretrained HuggingFace models
on the **same shared test set** used in notebook 04.

**Fixes applied from advisor review:**
- Error 2 fixed: all models evaluated on the same 1,030 test rows
  (loaded from test_texts.csv — no more tail slice of drifted CSV)
- Error 2 fixed: LSTM score computed from metrics.json — not hardcoded
- Error 2 fixed: zero-shot caveat added — HuggingFace models never saw
  our training data or our label mapping rules
- Debug cell removed

**Important caveat (stated with every result):**
Our LSTM was trained on this exact distribution using our star-to-class
labeling rule. The three HuggingFace models are evaluated zero-shot —
they were never trained on our data or our label definitions. The Neutral
class (exactly 3 stars) is an artifact of our labeling rule and is
essentially unknowable to a zero-shot model. This comparison demonstrates
domain-specific training value, not raw model superiority.

Before running: make sure notebooks 02, 03 and 04 have been run first.

In [1]:
# ── CELL 1: Import libraries ──
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))
from src.config import *

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ── CELL 2: Load shared test set from test_texts.csv ──
# Error 2 fix: load the same 1,030 test rows saved at split time in notebook 02
# All models — LSTM and HuggingFace — are evaluated on these exact rows
# No more tail slice of a drifted CSV file

df_test = pd.read_csv('../data/test_texts.csv')

label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df_test['label'] = df_test['true_sentiment'].map(label_map)
y_true   = df_test['label'].values
reviews  = df_test['verified_reviews'].tolist()

print(f"Shared test set loaded: {len(df_test)} rows")
print("\nDistribution:")
print(df_test['true_sentiment'].value_counts())
print("\nNote: these are the exact same rows evaluated in notebook 04")

Shared test set loaded: 1030 rows

Distribution:
true_sentiment
Negative    371
Neutral     361
Positive    298
Name: count, dtype: int64

Note: these are the exact same rows evaluated in notebook 04


In [3]:
# ── CELL 3: Load LSTM results from metrics.json ──
# Error 2 fix: LSTM scores loaded from committed metrics.json
# not hardcoded — provenance is guaranteed

with open('../models/metrics.json', 'r') as f:
    metrics = json.load(f)

lstm_acc     = metrics['test_accuracy']
lstm_f1      = metrics['test_macro_f1'] * 100
lstm_neg_f1  = metrics['per_class_f1']['Negative']
lstm_neu_f1  = metrics['per_class_f1']['Neutral']
lstm_pos_f1  = metrics['per_class_f1']['Positive']

print("LSTM results loaded from metrics.json:")
print(f"  Test Accuracy: {lstm_acc:.2f}%")
print(f"  Macro F1:      {lstm_f1:.2f}%")
print(f"  Negative F1:   {lstm_neg_f1:.4f}")
print(f"  Neutral F1:    {lstm_neu_f1:.4f}")
print(f"  Positive F1:   {lstm_pos_f1:.4f}")
print(f"  Seed:          {metrics['seed']}")
print(f"  n_test:        {metrics['n_test']}")

LSTM results loaded from metrics.json:
  Test Accuracy: 72.62%
  Macro F1:      72.63%
  Negative F1:   0.7467
  Neutral F1:    0.6396
  Positive F1:   0.7925
  Seed:          42
  n_test:        1030


In [4]:
# ── CELL 4: Load HuggingFace models ──
print("Loading Model 1: NLPTown...")
nlptown_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True, max_length=512)
print("NLPTown loaded!")

print("\nLoading Model 2: LiYuan Amazon...")
liyuan_pipeline = pipeline(
    "text-classification",
    model="LiYuan/amazon-review-sentiment-analysis",
    truncation=True, max_length=512)
print("LiYuan loaded!")

print("\nLoading Model 3: CardiffNLP RoBERTa...")
cardiff_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True, max_length=512)
print("CardiffNLP loaded!")
print("\nAll 3 models loaded successfully!")

Loading Model 1: NLPTown...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

NLPTown loaded!

Loading Model 2: LiYuan Amazon...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LiYuan loaded!

Loading Model 3: CardiffNLP RoBERTa...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CardiffNLP loaded!

All 3 models loaded successfully!


In [ ]:
# ── CELL 5: Run predictions on shared test set ──
print(f"Running predictions on {len(reviews)} shared test rows...")

print("Running NLPTown predictions...")
nlptown_results  = nlptown_pipeline(reviews,  batch_size=64)

print("Running LiYuan predictions...")
liyuan_results   = liyuan_pipeline(reviews,   batch_size=64)

print("Running CardiffNLP predictions...")
cardiff_results  = cardiff_pipeline(reviews,  batch_size=64)

print("All predictions complete!")

Running predictions on 1030 shared test rows...
Running NLPTown predictions...
Running LiYuan predictions...


In [ ]:
# ── CELL 6: Map outputs to our 3-class system ──
# NLPTown: outputs "1 star" to "5 stars"
def map_nlptown(result):
    stars = int(result['label'][0])
    if stars <= 2: return 0
    elif stars == 3: return 1
    else: return 2

# LiYuan: outputs "1 star" to "5 stars"
def map_liyuan(result):
    stars = int(result['label'].split(' ')[0])
    if stars <= 2: return 0
    elif stars == 3: return 1
    else: return 2

# CardiffNLP: outputs "negative", "neutral", "positive"
def map_cardiff(result):
    label = result['label'].lower()
    if label == 'negative': return 0
    elif label == 'neutral': return 1
    else: return 2

y_pred_nlptown  = np.array([map_nlptown(r)  for r in nlptown_results])
y_pred_liyuan   = np.array([map_liyuan(r)   for r in liyuan_results])
y_pred_cardiff  = np.array([map_cardiff(r)  for r in cardiff_results])

print("Label mapping complete!")

In [ ]:
# ── CELL 7: Calculate metrics for all HuggingFace models ──
label_names = ['Negative', 'Neutral', 'Positive']

nlptown_acc       = accuracy_score(y_true, y_pred_nlptown) * 100
nlptown_f1        = f1_score(y_true, y_pred_nlptown, average='macro') * 100
nlptown_f1_class  = f1_score(y_true, y_pred_nlptown, average=None)

liyuan_acc        = accuracy_score(y_true, y_pred_liyuan) * 100
liyuan_f1         = f1_score(y_true, y_pred_liyuan, average='macro') * 100
liyuan_f1_class   = f1_score(y_true, y_pred_liyuan, average=None)

cardiff_acc       = accuracy_score(y_true, y_pred_cardiff) * 100
cardiff_f1        = f1_score(y_true, y_pred_cardiff, average='macro') * 100
cardiff_f1_class  = f1_score(y_true, y_pred_cardiff, average=None)

print("=" * 75)
print(f"{'Model':<28} {'Accuracy':>10} {'Macro F1':>10} {'Neg F1':>8} {'Neu F1':>8} {'Pos F1':>8}")
print("-" * 75)
print(f"{'NLPTown':<28} {nlptown_acc:>9.2f}% {nlptown_f1:>9.2f}% {nlptown_f1_class[0]:>8.2f} {nlptown_f1_class[1]:>8.2f} {nlptown_f1_class[2]:>8.2f}")
print(f"{'LiYuan (Amazon)':<28} {liyuan_acc:>9.2f}% {liyuan_f1:>9.2f}% {liyuan_f1_class[0]:>8.2f} {liyuan_f1_class[1]:>8.2f} {liyuan_f1_class[2]:>8.2f}")
print(f"{'CardiffNLP (RoBERTa)':<28} {cardiff_acc:>9.2f}% {cardiff_f1:>9.2f}% {cardiff_f1_class[0]:>8.2f} {cardiff_f1_class[1]:>8.2f} {cardiff_f1_class[2]:>8.2f}")
print("=" * 75)

In [ ]:
# ── CELL 8: Full comparison table including LSTM ──
print("=" * 75)
print("FULL MODEL COMPARISON — Same 1,030 test rows, same label mapping")
print("=" * 75)
print(f"{'Model':<28} {'Accuracy':>10} {'Macro F1':>10} {'Neg F1':>8} {'Neu F1':>8} {'Pos F1':>8}")
print("-" * 75)
print(f"{'LensWord LSTM (trained)':<28} {lstm_acc:>9.2f}% {lstm_f1:>9.2f}% {lstm_neg_f1:>8.4f} {lstm_neu_f1:>8.4f} {lstm_pos_f1:>8.4f}")
print(f"{'NLPTown (zero-shot)':<28} {nlptown_acc:>9.2f}% {nlptown_f1:>9.2f}% {nlptown_f1_class[0]:>8.2f} {nlptown_f1_class[1]:>8.2f} {nlptown_f1_class[2]:>8.2f}")
print(f"{'LiYuan (zero-shot)':<28} {liyuan_acc:>9.2f}% {liyuan_f1:>9.2f}% {liyuan_f1_class[0]:>8.2f} {liyuan_f1_class[1]:>8.2f} {liyuan_f1_class[2]:>8.2f}")
print(f"{'CardiffNLP (zero-shot)':<28} {cardiff_acc:>9.2f}% {cardiff_f1:>9.2f}% {cardiff_f1_class[0]:>8.2f} {cardiff_f1_class[1]:>8.2f} {cardiff_f1_class[2]:>8.2f}")
print("=" * 75)
print()
print("IMPORTANT CAVEAT:")
print("LensWord LSTM was trained on this distribution using our star-to-class")
print("label rule. The three HuggingFace models are zero-shot — they were")
print("never trained on our data or our label definitions.")
print("The Neutral class (exactly 3 stars) is an artifact of our labeling")
print("rule and is essentially unknowable to a zero-shot model.")
print("This comparison shows the value of domain-specific training,")
print("not raw model superiority.")

In [ ]:
# ── CELL 9: Visualization ──
models_names = ['LensWord LSTM\n(trained)', 'NLPTown\n(zero-shot)',
                'LiYuan\n(zero-shot)', 'CardiffNLP\n(zero-shot)']
accuracies = [lstm_acc, nlptown_acc, liyuan_acc, cardiff_acc]
macro_f1s  = [lstm_f1,  nlptown_f1,  liyuan_f1,  cardiff_f1]

x     = np.arange(len(models_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy',  color='steelblue')
bars2 = ax.bar(x + width/2, macro_f1s,  width, label='Macro F1', color='orange')

ax.set_title('LensWord — Model Comparison (Same Test Set, Zero-Shot Caveat)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Score (%)')
ax.set_xticks(x)
ax.set_xticklabels(models_names)
ax.legend()
ax.set_ylim(0, 100)
ax.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../models/final_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart saved to models/final_model_comparison.png")

In [ ]:
# ── CELL 10: Save comparison results ──
comparison = {
    "test_set": "test_texts.csv",
    "n_test": len(y_true),
    "caveat": "HuggingFace models evaluated zero-shot. LSTM trained on this distribution.",
    "models": {
        "LensWord_LSTM": {
            "trained": True,
            "accuracy": round(lstm_acc, 2),
            "macro_f1": round(lstm_f1/100, 4),
            "neg_f1": round(lstm_neg_f1, 4),
            "neu_f1": round(lstm_neu_f1, 4),
            "pos_f1": round(lstm_pos_f1, 4)
        },
        "NLPTown": {
            "trained": False,
            "zero_shot": True,
            "accuracy": round(nlptown_acc, 2),
            "macro_f1": round(nlptown_f1/100, 4),
            "neg_f1": round(float(nlptown_f1_class[0]), 4),
            "neu_f1": round(float(nlptown_f1_class[1]), 4),
            "pos_f1": round(float(nlptown_f1_class[2]), 4)
        },
        "LiYuan": {
            "trained": False,
            "zero_shot": True,
            "accuracy": round(liyuan_acc, 2),
            "macro_f1": round(liyuan_f1/100, 4),
            "neg_f1": round(float(liyuan_f1_class[0]), 4),
            "neu_f1": round(float(liyuan_f1_class[1]), 4),
            "pos_f1": round(float(liyuan_f1_class[2]), 4)
        },
        "CardiffNLP": {
            "trained": False,
            "zero_shot": True,
            "accuracy": round(cardiff_acc, 2),
            "macro_f1": round(cardiff_f1/100, 4),
            "neg_f1": round(float(cardiff_f1_class[0]), 4),
            "neu_f1": round(float(cardiff_f1_class[1]), 4),
            "pos_f1": round(float(cardiff_f1_class[2]), 4)
        }
    }
}

with open('../models/comparison_results.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print("Comparison results saved to models/comparison_results.json")
print(json.dumps(comparison, indent=2))